# labels.csv EDA

Bu notebook `data/labels.csv` için karar odaklı EDA üretir. Kod hücreleri bağımsız, kısa ve standart kütüphane ağırlıklıdır.


## 1. Kurulum ve veri yolu

Amaç: Proje kökünü ve kaynak CSV yolunu tek yerde tanımlamak.


In [1]:
from pathlib import Path
import csv, json, math, statistics, collections, datetime, os

ROOT = Path('/home/furkan/Projects/Yefai')
CSV_PATH = ROOT / 'data' / 'labels.csv'
REPORT_DIR = ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)
CSV_PATH


## 2. Veri okuma ve temel şema

Amaç: CSV ingest bütünlüğünü, satır/kolon sayısını ve kolon listesini doğrulamak.


In [2]:
with CSV_PATH.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    rows = list(reader)

print('shape:', (len(rows), len(columns)))
print('columns:', columns)
print('first_row:', rows[0])


shape: (1803, 11)
columns: ['ImageName', 'SensorName', 'Set', 'ImageID', 'SensorID', 'wear', 'type', 'ImageDateTime', 'SensorDateTime', 'ImageFile', 'SensorFile']
first_row: {'ImageName': 'File_name_2022-09-09T13_42_21.698185.jpg', 'SensorName': 'File_name_2022-09-09T13_30_37.534347.csv', 'Set': '1', 'ImageID': '0.0', 'SensorID': '0.0', 'wear': '30.0', 'type': 'flank_wear', 'ImageDateTime': '2022-09-09 13:42:21.698185', 'SensorDateTime': '2022-09-09 13:30:37.534347', 'ImageFile': 'MATWI/Set1/images/File_name_2022-09-09T13_42_21.698185.jpg', 'SensorFile': 'MATWI/Set1/sensordata/File_name_2022-09-09T13_30_37.534347.csv'}


## 3. Savunmacı kalite kontrolleri

Amaç: Eksik değerleri, benzersiz değerleri, duplicate satırları ve ID tutarlılığını modellemeden önce denetlemek.


In [3]:
def is_missing(v):
    return v is None or str(v).strip() == ''

def to_float(v):
    try:
        return float(v) if not is_missing(v) else None
    except Exception:
        return None

n = len(rows)
missing = {c: sum(is_missing(r.get(c)) for r in rows) for c in columns}
unique = {c: len(set(r.get(c, '') for r in rows if not is_missing(r.get(c)))) for c in columns}
exact_duplicates = n - len({tuple(r.get(c, '') for c in columns) for r in rows})
id_mismatch = sum(to_float(r['ImageID']) != to_float(r['SensorID']) for r in rows)

quality_rows = [(c, missing[c], f'{missing[c]/n:.1%}', unique[c]) for c in columns]
print('exact_duplicates:', exact_duplicates)
print('image_sensor_id_mismatch_rows:', id_mismatch)
for item in quality_rows:
    print(item)


exact_duplicates: 0
image_sensor_id_mismatch_rows: 1386
('ImageName', 122, '6.8%', 1680)
('SensorName', 103, '5.7%', 1700)
('Set', 0, '0.0%', 17)
('ImageID', 122, '6.8%', 152)
('SensorID', 103, '5.7%', 213)
('wear', 140, '7.8%', 28)
('type', 135, '7.5%', 3)
('ImageDateTime', 122, '6.8%', 1680)
('SensorDateTime', 103, '5.7%', 1700)
('ImageFile', 122, '6.8%', 1680)
('SensorFile', 103, '5.7%', 1700)


## 4. Hedef ve grup dağılımları

Amaç: `type`, `wear` ve `Set` dağılımlarını imbalance ve validation kararları için okumak.


In [4]:
type_counts = collections.Counter(r['type'] or '<missing>' for r in rows)
wear_counts = collections.Counter(r['wear'] or '<missing>' for r in rows)
set_counts = collections.Counter(r['Set'] or '<missing>' for r in rows)

print('type_counts')
for k, v in type_counts.most_common():
    print(k, v, f'{v/n:.1%}')
print('\nwear_counts')
for k, v in sorted(wear_counts.items(), key=lambda kv: float(kv[0]) if kv[0] != '<missing>' else -1):
    print(k, v, f'{v/n:.1%}')
print('\nset_counts')
for k, v in sorted(set_counts.items(), key=lambda kv: float(kv[0]) if kv[0].replace('.', '', 1).isdigit() else 9999):
    print(k, v, f'{v/n:.1%}')


type_counts
flank_wear 1154 64.0%
flank_wear+adhesion 335 18.6%
adhesion 179 9.9%
<missing> 135 7.5%

wear_counts
<missing> 140 7.8%
15.0 32 1.8%
30.0 150 8.3%
45.0 172 9.5%
50.0 8 0.4%
60.0 212 11.8%
75.0 145 8.0%
90.0 268 14.9%
100.0 39 2.2%
105.0 45 2.5%
120.0 143 7.9%
125.0 13 0.7%
135.0 13 0.7%
150.0 109 6.0%
165.0 2 0.1%
175.0 3 0.2%
180.0 102 5.7%
200.0 9 0.5%
210.0 48 2.7%
240.0 43 2.4%
250.0 3 0.2%
270.0 35 1.9%
300.0 36 2.0%
330.0 1 0.1%
360.0 12 0.7%
390.0 1 0.1%
420.0 17 0.9%
450.0 1 0.1%
750.0 1 0.1%

set_counts
1 98 5.4%
2 115 6.4%
3 106 5.9%
4 108 6.0%
5 73 4.0%
6 214 11.9%
7 107 5.9%
8 104 5.8%
9 104 5.8%
10 104 5.8%
11 104 5.8%
12 55 3.1%
13 54 3.0%
14 158 8.8%
15 51 2.8%
16 101 5.6%
17 147 8.2%


## 5. Sayısal kolon özeti

Amaç: `Set`, ID ve `wear` kolonlarının aralık, medyan, ortalama ve yayılımını çıkarmak.


In [5]:
def quantile(vals, p):
    vals = sorted(vals)
    idx = (len(vals) - 1) * p
    lo, hi = math.floor(idx), math.ceil(idx)
    return vals[lo] if lo == hi else vals[lo] * (hi - idx) + vals[hi] * (idx - lo)

numeric_columns = ['Set', 'ImageID', 'SensorID', 'wear']
numeric_stats = {}
for c in numeric_columns:
    vals = [to_float(r[c]) for r in rows]
    vals = [v for v in vals if v is not None and math.isfinite(v)]
    numeric_stats[c] = {
        'count': len(vals),
        'min': min(vals),
        'q25': quantile(vals, .25),
        'median': quantile(vals, .50),
        'mean': sum(vals) / len(vals),
        'q75': quantile(vals, .75),
        'max': max(vals),
        'std': statistics.pstdev(vals),
    }

for c, s in numeric_stats.items():
    print(c, {k: round(v, 3) if isinstance(v, float) else v for k, v in s.items()})


Set {'count': 1803, 'min': 1.0, 'q25': 5.0, 'median': 8.0, 'mean': 8.809, 'q75': 14.0, 'max': 17.0, 'std': 4.94}
ImageID {'count': 1681, 'min': 0.0, 'q25': 24.0, 'median': 49.0, 'mean': 53.55, 'q75': 79.0, 'max': 151.0, 'std': 35.426}
SensorID {'count': 1700, 'min': 0.0, 'q25': 24.75, 'median': 49.5, 'mean': 56.872, 'q75': 81.0, 'max': 212.0, 'std': 41.975}
wear {'count': 1663, 'min': 15.0, 'q25': 60.0, 'median': 90.0, 'mean': 109.528, 'q75': 150.0, 'max': 750.0, 'std': 77.977}


## 6. Zaman hizalaması ve dosya varlığı

Amaç: Görüntü ve sensör zamanları arasındaki farkı, path kökünü ve dosya manifest tutarlılığını denetlemek.


In [6]:
def parse_dt(v):
    try:
        return datetime.datetime.fromisoformat(v)
    except Exception:
        return None

def stat(vals):
    vals = sorted(vals)
    return {
        'count': len(vals), 'min': min(vals), 'q25': quantile(vals, .25),
        'median': quantile(vals, .50), 'mean': sum(vals)/len(vals),
        'q75': quantile(vals, .75), 'max': max(vals),
    }

image_dts = [parse_dt(r['ImageDateTime']) for r in rows]
sensor_dts = [parse_dt(r['SensorDateTime']) for r in rows]
deltas = [(i - s).total_seconds() / 60 for i, s in zip(image_dts, sensor_dts) if i and s]
delta_stats = stat(deltas)
image_exists_data = sum((ROOT / 'data' / r['ImageFile']).exists() for r in rows)
sensor_exists_data = sum((ROOT / 'data' / r['SensorFile']).exists() for r in rows)
image_exists_root = sum((ROOT / r['ImageFile']).exists() for r in rows)
sensor_exists_root = sum((ROOT / r['SensorFile']).exists() for r in rows)

print('ImageDateTime:', min(d for d in image_dts if d), '->', max(d for d in image_dts if d))
print('SensorDateTime:', min(d for d in sensor_dts if d), '->', max(d for d in sensor_dts if d))
print('image_minus_sensor_minutes:', {k: round(v, 3) if isinstance(v, float) else v for k, v in delta_stats.items()})
print('exists under data:', {'images': (image_exists_data, n), 'sensors': (sensor_exists_data, n)})
print('exists under project root:', {'images': (image_exists_root, n), 'sensors': (sensor_exists_root, n)})


ImageDateTime: 2022-09-09 13:42:21.698185 -> 2023-07-03 14:34:18.507821
SensorDateTime: 2022-09-09 13:30:37.534347 -> 2023-07-03 14:32:39.087120
image_minus_sensor_minutes: {'count': 1578, 'min': 1.306, 'q25': 1.382, 'median': 1.456, 'mean': 4.29, 'q75': 1.676, 'max': 3993.488}
exists under data: {'images': (122, 1803), 'sensors': (103, 1803)}
exists under project root: {'images': (122, 1803), 'sensors': (103, 1803)}


## 7. Çapraz tablolar

Amaç: Set/type/wear ilişkilerini validation ve segment bazlı hata analizi için görünür yapmak.


In [7]:
def crosstab(row_key, col_key):
    table = collections.defaultdict(collections.Counter)
    for r in rows:
        table[r[row_key] or '<missing>'][r[col_key] or '<missing>'] += 1
    return {k: dict(v) for k, v in sorted(table.items(), key=lambda kv: kv[0])}

set_type = crosstab('Set', 'type')
type_wear = crosstab('type', 'wear')
set_wear = crosstab('Set', 'wear')

print('set_type')
for k, v in set_type.items():
    print(k, v)
print('\ntype_wear')
for k, v in type_wear.items():
    print(k, v)


set_type
1 {'flank_wear': 73, 'adhesion': 19, '<missing>': 6}
10 {'flank_wear': 94, 'adhesion': 8, '<missing>': 2}
11 {'adhesion': 5, 'flank_wear': 96, '<missing>': 3}
12 {'flank_wear+adhesion': 50, '<missing>': 5}
13 {'flank_wear+adhesion': 46, 'flank_wear': 7, '<missing>': 1}
14 {'flank_wear': 118, 'adhesion': 34, '<missing>': 6}
15 {'flank_wear': 21, 'adhesion': 29, '<missing>': 1}
16 {'flank_wear+adhesion': 96, 'flank_wear': 3, '<missing>': 2}
17 {'flank_wear': 77, 'flank_wear+adhesion': 70}
2 {'flank_wear': 63, 'adhesion': 34, '<missing>': 18}
3 {'<missing>': 2, 'flank_wear+adhesion': 73, 'flank_wear': 31}
4 {'<missing>': 9, 'flank_wear': 86, 'adhesion': 13}
5 {'flank_wear': 60, 'adhesion': 5, '<missing>': 8}
6 {'flank_wear': 140, 'adhesion': 8, '<missing>': 66}
7 {'flank_wear': 96, 'adhesion': 9, '<missing>': 2}
8 {'flank_wear': 89, '<missing>': 2, 'adhesion': 13}
9 {'flank_wear': 100, 'adhesion': 2, '<missing>': 2}

type_wear
<missing> {'<missing>': 135}
adhesion {'60.0': 28, '9

## 8. Makine-okunabilir özet üretimi

Amaç: Notebook çıktılarından rapor/otomasyon için tekrar kullanılabilir JSON özeti yazmak.


In [8]:
summary = {
    'shape': [n, len(columns)],
    'columns': columns,
    'missing': missing,
    'unique': unique,
    'exact_duplicates': exact_duplicates,
    'image_sensor_id_mismatch_rows': id_mismatch,
    'numeric_stats': numeric_stats,
    'type_counts': dict(type_counts),
    'wear_counts': dict(wear_counts),
    'set_counts': dict(set_counts),
    'set_type': set_type,
    'type_wear': type_wear,
    'set_wear': set_wear,
    'image_minus_sensor_minutes': delta_stats,
    'file_existence_under_data': {'images': [image_exists_data, n], 'sensors': [sensor_exists_data, n]},
    'file_existence_under_project_root': {'images': [image_exists_root, n], 'sensors': [sensor_exists_root, n]},
}
summary_path = REPORT_DIR / 'labels_eda_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print(summary_path)


/home/furkan/Projects/Yefai/reports/labels_eda_summary.json


## 9. Çalıştırılmış EDA sonucu ve kararlar

Bu bölüm, bu çalıştırmada üretilen gerçek özetlerden dolduruldu.

- Veri şekli: `1803` satır x `11` kolon.
- Eksik değerler: ImageName/ImageID/ImageDateTime/ImageFile `122`, SensorName/SensorID/SensorDateTime/SensorFile `103`, wear `140`, type `135`.
- Exact duplicate: `0`.
- ImageID/SensorID uyuşmazlığı: tüm satırlarda `1386`; iki ID de mevcutken `1161` satır. Bu genellikle görüntü-sensör eşlemesinin birebir aynı ID ile yapılmadığını gösterir.
- `type` sınıf dağılımı dengesiz; en büyük sınıf `flank_wear` = `1154` satır.
- `wear` aralığı: `15` - `750`, medyan `90`.
- Image/Sensor zaman farkı: medyan `1.456` dakika, max `3993.488` dakika.
- Dosya yolları `data/` altında tam görünüyor: ImageFile `122/1803`, SensorFile `103/1803`.

### Validation / leakage kararı

`Set`, dosya adları, ID ve timestamp alanları deney sekansını kodladığı için ham feature olarak kullanılmamalı. İlk güvenilir validation adayı Set bazlı veya zaman-sıralı holdout olmalı; random split ancak leakage denetimi sonrası karşılaştırma baseline'ı olarak kullanılmalı.

### Preprocessing kararı

- Metadata kolonları (`ImageName`, `SensorName`, `ImageFile`, `SensorFile`, `ImageID`, `SensorID`, timestamp) split/audit için korunmalı; doğrudan model feature'ı yapılmamalı.
- `type` için macro-F1/balanced accuracy; `wear` için ordinal/regression metrikleri birlikte raporlanmalı.
- Asıl model sinyali görüntü ve sensör dosyalarından üretilecek feature'larda aranmalı; bu CSV tek başına manifest/label tablosudur.

### Sonraki deneyler

1. Set bazlı holdout ile görüntü-only, sensör-only ve fusion baseline kur.
2. `wear` ordinal yapısını koruyan loss/metric seçeneklerini karşılaştır.
3. Type sınıf dengesizliği için class weight ve segment bazlı hata analizi ekle.
